In [22]:
import pandas as pd
import numpy as np

df_tow = pd.read_parquet("dane/interim/towar_columns_selected-records_full.parquet")
df_asort = pd.read_parquet("dane/interim/asort_columns_selected-records_full.parquet")
df_dok = pd.read_parquet("dane/interim/dok_columns_selected-records_full.parquet")
df_pozd = pd.read_parquet("dane/interim/pozdok_columns_selected-records_full.parquet")

print(df_tow.shape, df_asort.shape, df_dok.shape, df_pozd.shape)

(41378, 7) (337, 2) (952122, 9) (4272656, 9)


In [2]:
# TODO: opcjonalnie po wersji MVP, jako jedna z poprawek - aktualizacja tabeli towarów (więcej SKU)
# df_tow = pd.read_csv(
#     "dane/Towar_2026.csv",
#     encoding='utf-8-sig',  # obsługuje BOM
#     sep=';',
#     decimal=",",
#     on_bad_lines='skip'
# )
# print(df_towar_new.shape)
# print(df_towar_new['TowId'].max())
# print(df_towar_new[df_towar_new['TowId'] > 81360]['TowId'].nunique())
# df_towar_new[df_towar_new['TowId']>81360]['Nazwa']

In [23]:
print("df_tow TowId:", df_tow['TowId'].dtype)
print("df_dok DokId:", df_dok['DokId'].dtype)
print("df_pozd TowId:", df_pozd['TowId'].dtype)
print("df_pozd DokId:", df_pozd['DokId'].dtype)
print("df_asort AsId:", df_asort['AsId'].dtype)
print("df_tow AsId:", df_tow['AsId'].dtype)

df_tow TowId: int64
df_dok DokId: int64
df_pozd TowId: int64
df_pozd DokId: int64
df_asort AsId: int64
df_tow AsId: int64


In [24]:
# Sprawdzenie duplikatow na kluczach laczenia (ryzyko fan-out przy merge)
print("Duplikaty DokId w df_dok:", df_dok['DokId'].duplicated().sum())
print("Duplikaty TowId w df_tow:", df_tow['TowId'].duplicated().sum())
print("Duplikaty AsId w df_asort:", df_asort['AsId'].duplicated().sum())

Duplikaty DokId w df_dok: 0
Duplikaty TowId w df_tow: 0
Duplikaty AsId w df_asort: 0


In [25]:
# Krok 1: PozDok + Dok
dok_pozd = df_pozd.merge(df_dok, on='DokId', how='inner')
print(f"dok_pozd: {dok_pozd.shape}")

# Krok 2: + Towar
dok_pozd_tow = dok_pozd.merge(df_tow, on='TowId', how='inner')
print(f"dok_pozd_tow: {dok_pozd_tow.shape}")

# Krok 3: + Asort
fact_inka = dok_pozd_tow.merge(df_asort, on='AsId', how='inner')
print(f"fact_inka: {fact_inka.shape}")

dok_pozd: (4272656, 17)
dok_pozd_tow: (4261678, 23)
fact_inka: (4261678, 24)


In [17]:
# # Diagnostyka utraconych wierszy przy inner join (porownanie z left join + indicator)
# check1 = df_pozd.merge(df_dok, on='DokId', how='left', indicator=True)
# print("Krok 1 (PozDok+Dok) - bez dopasowania:", (check1['_merge'] == 'left_only').sum())

# check2 = dok_pozd.merge(df_tow, on='TowId', how='left', indicator=True)
# print("Krok 2 (+Towar) - bez dopasowania:", (check2['_merge'] == 'left_only').sum())

# check3 = dok_pozd_tow.merge(df_asort, on='AsId', how='left', indicator=True)
# print("Krok 3 (+Asort) - bez dopasowania:", (check3['_merge'] == 'left_only').sum())

In [18]:
# # Analiza brakujacych TowId w Kroku 2 (dok_pozd + Towar)
# missing_tow_ids = check2[check2['_merge'] == 'left_only']['TowId'].unique()
# print("Liczba unikalnych brakujacych TowId:", len(missing_tow_ids))
# print(missing_tow_ids[:20])

# # ile transakcji/wartosci to dotyczy
# print("Suma Wartosc dla utraconych wierszy:", check2[check2['_merge'] == 'left_only']['Wartosc'].sum())

In [26]:
doc_type_map = pd.DataFrame([
    (2,   'PZ',             1, True,  'IP',           +1, 'przyjecie', False),
    (8,   'ZWPAR',          1, True,  'IP',           +1, 'zwrot',     False),
    (9,   'PW',             1, True,  'IP',           +1, 'przyjecie', False),  # ← NOWY jak PZ
   
    (10,  'RW',             4, True,  'IP',           -1, 'rozchod',   False),
    (21,  'DF',             4, True,  'IP',           -1, 'sprzedaz',  False),
    (23,  'ST',             4, True,  'IP',           -1, 'strata',    True),
    
    (14,  'BO',             3, True,  'RESET_IP',     +1, 'bo',        False),
    (16,  'REM',            3, True,  'RESET_IP',     +1, 'remanent',  False),

    (26,  'ROZB',           1, True,  'IP_IM_DELTA',  +1, 'rozbieznosc',      False),
    (78,  'PRZES',          1, True,  'IP_IM_DELTA',  +1, 'przesuniecie',     False),

    (88,  'PRZES_GR',       2, True,  'IP_IM_DELTA',  +1, 'przesuniecie',     False),
    
    (1,   'OPAK',           2, False, 'Brak',         0, 'neutralny', False),
    (4,   'ZWFD',           4, False, 'Brak',         0, 'neutralny', False),
    (18,  'PRZEC',          6, False, 'Brak',         0, 'przecena',  True),
    (19,  'ZAMR_PRZEC',     0, False, 'Brak',         0, 'przecena',  True),
    (30,  'B_OPAK',         0, False, 'Brak',         0, 'neutralny', False),
    (33,  'FV',             0, False, 'Brak',         0, 'neutralny', False),
    (50,  'ZAM',            0, False, 'Brak',         0, 'neutralny', False),
    (59,  'DF',             0, False, 'Brak',         0, 'neutralny', False),
    (60,  'XXX',            0, False, 'Brak',         0, 'neutralny', False),
    (81,  'CDPRS',          1, False, 'Brak',         0, 'prasa',     False),
    (82,  'CRPRS',          7, False, 'Brak',         0, 'prasa',     False),
    (100, 'ZM_ST_VAT',      1, False, 'Brak',         0, 'neutralny', False),
    (100, 'ZM_ST_VAT',      4, False, 'Brak',         0, 'neutralny', False),
    (126, 'PLAN_ZM_ST_VAT', 0, False, 'Brak',         0, 'neutralny', False),
    (900, 'ST_rozli',       4, False, 'Brak',         0, 'neutralny', False),
    (900, 'ZWPAR_rozli',    1, False, 'Brak',         0, 'neutralny', False),
    (981, 'MM_INW_MOB',     0, False, 'Brak',         0, 'neutralny', False),  # ← NOWY
], columns=['TypDok', 'Dokument', 'TypPoz', 'WplywNaStan',
            'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane'])

In [27]:
fact_inka

,DokId,Kolejnosc,NrPozycji,TowId,TypPoz,IloscPlus,IloscMinus,CenaPoRab,Wartosc,Data,...,Razem,DoZaplaty,Zaplacono,AsId,NazwaTow,EAN,Opis1,Producent,AktywnyTow,NazwaAsort
0,41775,2,1,19623,0,1.000,0.0,917.1500,917.1500,2018-09-14,...,1128.09,1128.09,1128.09,191,ZAPALNICZKA DAVIDOFF ŻAR SZT,2.222000e+03,nan,NaN,1,ZAPALNICZKI I ZAPAŁKI /przemysłowe
1,131173,1,1,23638,0,1.000,0.0,112.5000,112.5000,2019-01-07,...,138.38,138.38,138.38,254,USŁUGA MARKETINGOWA,2.610000e+02,nan,NaN,1,xx KOSZTY / RABATY
2,142162,1,1,23638,0,1.000,0.0,75.0000,75.0000,2019-01-21,...,92.25,92.25,92.25,254,USŁUGA MARKETINGOWA,2.610000e+02,nan,NaN,1,xx KOSZTY / RABATY
3,159723,1,1,23638,0,1.000,0.0,7956.8100,7956.8100,2019-02-11,...,9786.88,9786.88,9786.88,254,USŁUGA MARKETINGOWA,2.610000e+02,nan,NaN,1,xx KOSZTY / RABATY
4,162945,1,1,25390,0,1.000,0.0,100.0000,100.0000,2019-02-22,...,123.00,123.00,123.00,216,USŁUGA PROMOCYJNA SŁODYCZE Q1,1.111110e+05,nan,NaN,0,x SŁODYCZE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4261673,2237660,11,11,49495,4,1.000,0.0,5.2286,5.2286,2026-01-31,...,96.23,96.23,96.23,70,POLĘDWICA SOPOCKA 100G BALCERZAK,5.906776e+12,nan,NaN,1,WĘDLINY PACZKOWANE
4261674,2237660,12,12,66214,4,1.000,0.0,6.6571,6.6571,2026-01-31,...,96.23,96.23,96.23,70,PIERŚ Z INDYKA 100G BALCERZAK,5.906776e+12,nan,NaN,1,WĘDLINY PACZKOWANE
4261675,2237660,16,16,52905,4,1.000,0.0,2.4309,2.4309,2026-01-31,...,96.23,96.23,96.23,97,"NAPÓJ ENER BLACK STRAWBERRY 0,25L FOODCARE",5.900552e+12,nan,NaN,1,ENERGETYKI I IZOTONIKI
4261676,2237660,17,17,80379,4,1.000,0.0,15.4390,15.4390,2026-01-31,...,96.23,96.23,96.23,322,PAPIEROSY KENT MODE GREEN MARINE 20SZT BAT,5.905659e+12,nan,NaN,1,PAPIEROSY


In [20]:
doc_type_map

,TypDok,Dokument,TypPoz,WplywNaStan,MetodaLiczenia,Mnoznik,TypRuchu,CzyNiechciane
0,2,PZ,1,True,IP,1,przyjecie,False
1,8,ZWPAR,1,True,IP,1,zwrot,False
2,9,PW,1,True,IP,1,przyjecie,False
3,10,RW,4,True,IP,-1,rozchod,False
4,21,DF,4,True,IP,-1,sprzedaz,False
5,23,ST,4,True,IP,-1,strata,True
6,14,BO,3,True,RESET_IP,1,bo,False
7,16,REM,3,True,RESET_IP,1,remanent,False
8,26,ROZB,1,True,IP_IM_DELTA,1,rozbieznosc,False
9,78,PRZES,1,True,IP_IM_DELTA,1,przesuniecie,False


In [28]:
fact_inka_merge = fact_inka.merge(doc_type_map, on=['TypDok', 'TypPoz'], how='left')

In [29]:
fact_inka_merge_data = fact_inka_merge[fact_inka_merge['Data'].dt.year.isin([2023, 2024, 2025, 2026])]
print(f"Po filtrze dat: {fact_inka_merge_data.shape}")

Po filtrze dat: (4260543, 30)


In [30]:
fact_inka_merge_data.to_parquet("dane/interim/fact_inka_records_2023_2026.parquet", compression='zstd',index=False)
print(f"Zapisano: {fact_inka_merge_data.shape}")

Zapisano: (4260543, 30)


In [9]:
fact_inka_merge.to_parquet("dane/interim/fact_inka-records_full.parquet", index=False)
print(f"Zapisano: {fact_inka_merge.shape}")

Zapisano: (4261678, 35)
